# Bloom Filter for Malicious URL Detection
### Probability and Statistics | B.Tech Computer Science and Engineering

## 1. Project introduction and objectives

This notebook studies Bloom Filter membership testing using the project's training and held-out URL datasets, implementation, saved configuration experiments, and lookup benchmark.

**Objectives**
- Derive the Bloom Filter bit-array size and hash-count equations.
- Calculate bit occupancy and the theoretical false-positive probability.
- Compare calculated theoretical results with previously measured experiment results.
- Examine lookup-time and memory trade-offs against Python Set and SQLite.

**Interpretation and safety:** All URL values are treated as strings. No URL is opened, visited, or requested. A Bloom Filter positive means **possibly present**, not confirmed membership; false positives are possible.

## 2. Dataset description

`data/clean_urls.csv` contains the 100,000 defanged training strings inserted into the filter. `data/test_urls.csv` contains 74,790 held-out strings from the complementary split. The held-out split is disjoint from training, so a positive for a held-out test item is counted as a false positive in the project's experiment.

The notebook reads existing CSV results and uses the existing implementation. It does not rerun scripts that overwrite project files.

## 3. Import required Python libraries

The analysis uses Python's standard library, pandas, and matplotlib. It imports the existing `BloomFilter` implementation from `src/bloom_filter.py`. A Jupyter Python kernel is required to run the notebook interactively. `pandas` is listed in the project's `requirements.txt`; if needed, install notebook-specific dependencies with `python -m pip install matplotlib jupyter ipykernel`.

In [ ]:
from pathlib import Path
import math
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Support launching from either the project root or its notebooks/ directory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data").is_dir():
    raise FileNotFoundError("Could not find the project data/ directory from the current working directory.")

DATA_DIR = PROJECT_ROOT / "data"
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from bloom_filter import BloomFilter

print("Project root:", PROJECT_ROOT.resolve())
print("Libraries: pandas", pd.__version__, "| matplotlib", plt.matplotlib.__version__)

## 4. Load and inspect training and testing datasets

The next cell reads the URL columns as strings, checks the expected row counts and headers, checks for blanks and overlap, and prints a few stored examples as text only.

In [ ]:
# Load the datasets as text only and verify the expected schema and sizes.
training_df = pd.read_csv(DATA_DIR / "clean_urls.csv", dtype={"url": str}, keep_default_na=False)
testing_df = pd.read_csv(DATA_DIR / "test_urls.csv", dtype={"url": str}, keep_default_na=False)

assert list(training_df.columns) == ["url"]
assert list(testing_df.columns) == ["url"]
assert len(training_df) == 100_000
assert len(testing_df) == 74_790
assert training_df["url"].str.strip().ne("").all()
assert testing_df["url"].str.strip().ne("").all()
assert not set(training_df["url"]).intersection(testing_df["url"])

training_urls = training_df["url"].str.strip().tolist()
testing_urls = testing_df["url"].str.strip().tolist()
print(f"Training records: {len(training_urls):,}")
print(f"Testing records:  {len(testing_urls):,}")
print("Training/testing overlap: 0")
print("Example stored strings (not opened):")
for value in training_urls[:3]:
    print(" -", value)

## 5. Bloom Filter mathematical theory

A Bloom Filter stores membership information in a bit array of size $m$ and uses $k$ hash functions for each of $n$ inserted items. It is space-efficient and supports fast membership checks, but does not store the original items and can produce false positives.

For target false-positive probability $p$, the conventional optimal design equations are:

$$m = -\frac{n\ln(p)}{(\ln 2)^2}$$

$$k = \frac{m}{n}\ln 2$$

The implementation rounds $m$ upward to an integer and uses `round()` for $k$. Therefore its integer hash count can differ slightly from the real-valued optimum.

## 6-7. Calculate the optimal bit-array size and hash count

Using the training count and 1% target probability, the next cell evaluates the equations in Section 5, rounds to the implementation's integer choices, and confirms those choices against the existing class. The real-valued optimal hash count and implemented integer count are reported separately.

In [ ]:
# Calculate the design from the stated active target and actual training count.
n = len(training_urls)
target_fpr = 0.01
m_calculated = math.ceil(-(n * math.log(target_fpr)) / (math.log(2) ** 2))
k_optimal_real = (m_calculated / n) * math.log(2)
k_implemented = round(k_optimal_real)

active_filter = BloomFilter(expected_items=n, false_positive_rate=target_fpr)
assert active_filter.size == m_calculated
assert active_filter.hash_count == k_implemented

print(f"n (training items): {n:,}")
print(f"Target p: {target_fpr:.2%}")
print(f"Calculated bit-array size m: {m_calculated:,} bits")
print(f"Calculated real-valued optimal k: {k_optimal_real:.6f}")
print(f"Implemented integer k: {k_implemented}")
print(f"Implementation bit-array memory: {active_filter.get_memory_usage() / 1024:.6f} KB")

## 8. Probability that bits remain unset or become set

Assuming approximately independent uniform hash positions, after $kn$ bit-setting operations the probability a particular bit remains zero is approximately

$$P_0 \approx \left(1-\frac{1}{m}\right)^{kn} \approx e^{-kn/m}.$$

The probability that the bit is set is $P_1 = 1-P_0$. Under the standard approximation, the theoretical false-positive probability is

$$P(FP) \approx P_1^k = \left(1-e^{-kn/m}\right)^k.$$

The project's `BloomFilter.get_theoretical_false_positive_rate()` uses this exponential approximation. The following cell labels computed formula values separately from experiment measurements.

## 9. Calculate theoretical false-positive probability

The next calculation applies the standard exponential approximation to the probability that all $k$ queried bit positions are set. The implementation-based theoretical rate is checked later after inserting the training URLs, so that its stored item count is correct.

In [ ]:
p_bit_zero_finite = (1 - 1 / m_calculated) ** (k_implemented * n)
p_bit_zero_exponential = math.exp(-(k_implemented * n) / m_calculated)
p_bit_set_finite = 1 - p_bit_zero_finite
p_bit_set_exponential = 1 - p_bit_zero_exponential
fpr_theoretical_formula = p_bit_set_exponential ** k_implemented
print(f"Calculated P(bit remains zero), finite form: {p_bit_zero_finite:.6f}")
print(f"Calculated P(bit remains zero), exponential approximation: {p_bit_zero_exponential:.6f}")
print(f"Calculated P(bit is set), finite form: {p_bit_set_finite:.6f}")
print(f"Calculated P(bit is set), exponential approximation: {p_bit_set_exponential:.6f}")
print(f"Calculated theoretical FPR (exponential formula): {100 * fpr_theoretical_formula:.6f}%")
print(f"Calculated theoretical FPR (formula): {100 * fpr_theoretical_formula:.6f}%")

## 10. Compare theoretical and observed false-positive rates

The next cell builds the filter using `src/bloom_filter.py` and tests the held-out split. Since the split is disjoint, held-out positives are false positives for this experiment. This computation uses URL strings only and does not make network requests.

**Previously measured result:** 785 false positives among 74,790 test URLs (1.0496%); theoretical rate 1.0039%. The cell below independently recomputes these values from the current input files.

In [ ]:
# Rebuild the active filter with the project's implementation and reproduce its held-out test.
active_filter = BloomFilter(expected_items=len(training_urls), false_positive_rate=0.01)
for url_text in training_urls:
    active_filter.add(url_text)

false_positives = sum(active_filter.contains(url_text) for url_text in testing_urls)
true_negatives = len(testing_urls) - false_positives
observed_fpr = false_positives / len(testing_urls)
implementation_theoretical_fpr = active_filter.get_theoretical_false_positive_rate()

print(f"Recomputed training items: {active_filter.item_count:,}")
print(f"Bit array: {active_filter.size:,} bits; hash functions: {active_filter.hash_count}")
print(f"Recomputed bit-array memory: {active_filter.get_memory_usage() / 1024:.6f} KB")
print(f"Recomputed held-out false positives: {false_positives:,}")
print(f"Recomputed held-out true negatives: {true_negatives:,}")
print(f"Recomputed observed FPR: {100 * observed_fpr:.6f}%")
print(f"Calculated theoretical FPR: {100 * implementation_theoretical_fpr:.6f}%")
assert false_positives == 785
assert true_negatives == 74_005
assert active_filter.size == 958_506
assert active_filter.hash_count == 7
assert math.isclose(100 * observed_fpr, 1.0496055622409413, rel_tol=0, abs_tol=1e-12)
assert math.isclose(100 * implementation_theoretical_fpr, 1.0039209581758124, rel_tol=0, abs_tol=1e-12)
assert math.isclose(implementation_theoretical_fpr, fpr_theoretical_formula, rel_tol=1e-12)

## 11. Analyze Bloom Filter configurations

`data/configuration_results.csv` contains the previously measured results for five target false-positive rates. The theoretical and observed columns below are **stored experiment results**, not estimates. The observed rate is based on testing the held-out URL set.

In [ ]:
config_df = pd.read_csv(DATA_DIR / "configuration_results.csv")
expected_config_columns = {"target", "bits", "hashes", "memory", "false_positives", "theoretical", "observed"}
assert expected_config_columns.issubset(config_df.columns)
assert len(config_df) == 5

config_display = config_df[["target", "bits", "hashes", "memory", "false_positives", "theoretical", "observed"]].copy()
config_display = config_display.rename(columns={
    "target": "Target FPR (%)", "bits": "Bits", "hashes": "Hashes",
    "memory": "Memory (KB)", "false_positives": "False positives",
    "theoretical": "Theoretical FPR (%)", "observed": "Observed FPR (%)"
})
config_display

## 12. Visualize theoretical versus observed rates

This grouped bar chart plots the theoretical and measured rates stored for each configuration. Values come from the existing experiment CSV.

In [ ]:
# Visualize stored theoretical and observed configuration results.
fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(config_df))
width = 0.36
ax.bar([i - width / 2 for i in x], config_df["theoretical"], width, label="Theoretical (stored)")
ax.bar([i + width / 2 for i in x], config_df["observed"], width, label="Observed (measured)")
ax.set_xticks(list(x), [f"{value:g}%" for value in config_df["target"]])
ax.set_xlabel("Target false-positive rate")
ax.set_ylabel("False-positive rate (%)")
ax.set_title("Theoretical vs. observed Bloom Filter false-positive rates")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

## 13. Analyze lookup benchmark results

The following timings are read from `data/benchmark_results.csv` and are previously measured results. They are reported in milliseconds per lookup. The comparison is machine- and run-dependent; the CSV is the source of the values shown here.

In [ ]:
benchmark_df = pd.read_csv(DATA_DIR / "benchmark_results.csv")
expected_benchmark_columns = {"Method", "Existing Lookup (ms)", "Absent Lookup (ms)"}
assert expected_benchmark_columns.issubset(benchmark_df.columns)
assert set(benchmark_df["Method"]) == {"Bloom Filter", "Python Set", "SQLite"}
benchmark_df

In [ ]:
# Compare timings on a logarithmic axis because the measured magnitudes differ substantially.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, column, title in zip(
    axes,
    ["Existing Lookup (ms)", "Absent Lookup (ms)"],
    ["Existing URL lookup", "Absent URL lookup"],
):
    bars = ax.bar(benchmark_df["Method"], benchmark_df[column], color=["#3B82A0", "#E2A84A", "#74839A"])
    ax.set_yscale("log")
    ax.set_title(title)
    ax.set_ylabel("Milliseconds per lookup (log scale)")
    ax.tick_params(axis="x", rotation=18)
    ax.grid(axis="y", which="both", alpha=0.25)
    for bar, value in zip(bars, benchmark_df[column]):
        ax.annotate(f"{value:.6f}", (bar.get_x() + bar.get_width() / 2, value),
                    xytext=(0, 4), textcoords="offset points", ha="center", fontsize=8)
fig.suptitle("Previously measured lookup benchmark results")
fig.tight_layout()
plt.show()

## 14. Compare Bloom Filter, Python Set, and SQLite

The timing benchmark shows Python Set faster than Bloom Filter in both existing-URL and absent-URL tests. Bloom Filter is valuable here for its compact bit-array representation, not because it won the measured timing comparison.

The cell below calculates approximate Python Set size at runtime from the loaded training strings and compares it with the Bloom Filter bytearray. SQLite is included as on-disk database file size. These are **calculated on this machine at notebook execution time** and are different kinds of measurements: Python Set and Bloom Filter are in-memory sizes; SQLite is a file size. Python object sizes may vary by Python version and platform.

In [ ]:
import os
import sys

python_url_set = set(training_urls)
python_set_bytes = sys.getsizeof(python_url_set) + sum(sys.getsizeof(value) for value in python_url_set)
bloom_bytes = active_filter.get_memory_usage()
sqlite_path = PROJECT_ROOT / "database" / "urlhaus.db"
sqlite_bytes = os.path.getsize(sqlite_path) if sqlite_path.is_file() else None

memory_comparison = pd.DataFrame([
    {"Method": "Bloom Filter", "Size": bloom_bytes / 1024, "Unit": "KB in memory", "Basis": "bytearray size via sys.getsizeof"},
    {"Method": "Python Set", "Size": python_set_bytes / 1024, "Unit": "KB in memory", "Basis": "set plus stored string objects"},
])
if sqlite_bytes is not None:
    memory_comparison.loc[len(memory_comparison)] = {
        "Method": "SQLite", "Size": sqlite_bytes / 1024, "Unit": "KB on disk", "Basis": "database file size"
    }

print("Calculated memory/storage comparison for this runtime:")
display(memory_comparison)
print("Python Set faster than Bloom Filter in both recorded timing tests:",
      benchmark_df.loc[benchmark_df["Method"] == "Python Set", "Existing Lookup (ms)"].iloc[0]
      < benchmark_df.loc[benchmark_df["Method"] == "Bloom Filter", "Existing Lookup (ms)"].iloc[0]
      and benchmark_df.loc[benchmark_df["Method"] == "Python Set", "Absent Lookup (ms)"].iloc[0]
      < benchmark_df.loc[benchmark_df["Method"] == "Bloom Filter", "Absent Lookup (ms)"].iloc[0])

## 15. Findings, limitations, and conclusion

### Findings
- The active design calculated from 100,000 training strings uses 958,506 bits and rounds the optimal real-valued hash count to 7.
- The project's measured active result is 785 false positives from 74,790 held-out strings (1.0496%), compared with a calculated theoretical rate of about 1.0039%.
- Across the stored benchmark, Python Set was faster than Bloom Filter for both existing and absent lookups. The Bloom Filter uses significantly less memory than a Python Set for this dataset.
- A Bloom Filter positive means **possibly present**; it is not proof of membership or of maliciousness. A negative means the inserted item is definitely not present under this Bloom Filter's standard assumptions.

### Limitations
- The theoretical probability relies on idealized hashing/independence assumptions; measurements depend on the dataset, implementation, machine, and run.
- Lookup timing figures are previously measured CSV values and are not regenerated by this notebook.
- Python Set and Bloom Filter memory estimates are process/runtime dependent. SQLite file size is disk storage and is not directly equivalent to in-memory usage.
- This project measures membership behavior; it is not a complete malware classification system. URLs are never contacted by this notebook.

### Conclusion
The experiment demonstrates the Bloom Filter's memory-efficiency trade-off and its probabilistic membership behavior. In the recorded lookup tests, Python Set is faster; Bloom Filter's main measured advantage is substantially lower memory usage.